## MC vs CYME: Bus Voltage & Transformer Configuration Comparison

#### Compare assigned bus voltages (`vn_kv`) and transformer configurations between the multiconductor
##### net objects and what CYME actually uses after the mc→CYME conversion.

### **Key differences to check:**
##### 1. **Bus base voltages** — MC stores explicit `vn_kv` per bus; CYME sets `UserDefinedBaseVoltage` only on transformer LV nodes
##### 2. **Transformer winding voltages** — MC converts L-L → winding voltage (÷√3 for Y, direct for D); CYME uses L-L directly
##### 3. **Connection type** — MC stores full vector group (from_phase/to_phase per circuit); CYME simplifies to `D_Yg` / `Yg_Yg`
##### 4. **Impedance split** — MC stores half-impedance per winding side; CYME uses total positive-sequence %Z
##### 5. **Power flow settings** — CYME disables line charging and voltage-sensitive loads by default -->

In [2]:
import pickle, math, os, time
from pathlib import Path
import pandas as pd
import numpy as np

PHASE_INT_TO_LETTER = {0: 'N', 1: 'A', 2: 'B', 3: 'C'}


def compare_bus_voltages_and_transformers(net, circuit_name):
    """Extract bus voltages and transformer configs from an mc net object
    and compute the equivalent CYME representation side-by-side."""

    # ------------------------------------------------------------------ #
    # PRE-COMPUTE LOOKUPS (avoid repeated DataFrame operations)          #
    # ------------------------------------------------------------------ #

    # -- Source bus lookup --
    source_buses = {}  # {bus_idx: vm_pu}
    for eg_name in ["ext_grid_sequence", "ext_grid"]:
        if eg_name in net and not net[eg_name].empty:
            eg = net[eg_name]
            if "bus" in eg.columns:
                for bus_idx in eg["bus"].unique():
                    mask = eg["bus"] == bus_idx
                    source_buses[int(bus_idx)] = float(eg.loc[mask, "vm_pu"].iloc[0])
            elif "bus" in eg.index.names:
                level = eg.index.names.index("bus")
                for bus_idx in eg.index.get_level_values(level).unique():
                    source_buses[int(bus_idx)] = 1.0

    # -- Transformer LV bus lookup (built once, not per-bus) --
    trafo_lv_map = {}  # {lv_bus_idx: trafo_name}
    trafo_flat_cache = {}  # {trafo_idx: (flat_df, buses, ordered_df)}
    if "trafo1ph" in net and not net["trafo1ph"].empty:
        trafo_indices = net.trafo1ph.index.get_level_values(0).unique()
        for t_idx in trafo_indices:
            t_data = net.trafo1ph.loc[t_idx]
            if isinstance(t_data, pd.Series):
                t_data = t_data.to_frame().T
            ordered_df = t_data.copy()
            flat = t_data.reset_index() if isinstance(t_data.index, pd.MultiIndex) else t_data

            if "bus" in flat.columns:
                bus_values = pd.to_numeric(flat["bus"], errors="coerce").dropna().astype(int).tolist()
                buses = sorted(set(bus_values))
            else:
                buses = sorted(set(flat.index.get_level_values("bus")))

            trafo_flat_cache[t_idx] = (flat, buses, ordered_df)
            if len(buses) >= 2:
                trafo_lv_map[buses[1]] = f"trafo_{t_idx}"

    # -- Bus name/vn_kv lookup (vectorized) --
    bus_indices = net.bus.index.get_level_values(0).unique()

    # ------------------------------------------------------------------ #
    # 1. BUS VOLTAGE TABLE                                                #
    # ------------------------------------------------------------------ #
    bus_rows = []
    for bus_idx in bus_indices:
        bus_data = net.bus.loc[bus_idx]
        if isinstance(bus_data, pd.DataFrame):
            name = bus_data["name"].iloc[0]
            vn_kv = float(bus_data["vn_kv"].iloc[0])
            phases = sorted(set(bus_data.index.get_level_values(-1)) - {0})
        else:
            name = bus_data["name"]
            vn_kv = float(bus_data["vn_kv"])
            phases = []

        phase_str = "".join(PHASE_INT_TO_LETTER.get(p, "") for p in phases)
        is_source = bus_idx in source_buses
        vm_pu = source_buses.get(bus_idx, 1.0)
        is_trafo_lv = bus_idx in trafo_lv_map
        trafo_name = trafo_lv_map.get(bus_idx, "")

        cyme_base_voltage = vn_kv if (is_trafo_lv or is_source) else "inherited/unset"
        cyme_operating_kv = round(vn_kv * vm_pu / math.sqrt(3), 6) if is_source else ""

        bus_rows.append(
            {
                "CKT_KEY": circuit_name,
                "BUS_IDX": bus_idx,
                "BUS_NAME": name,
                "PHASES": phase_str,
                "MC_VN_KV (L-L)": vn_kv,
                "IS_SOURCE": is_source,
                "IS_TRAFO_LV": is_trafo_lv,
                "TRAFO_NAME": trafo_name,
                "CYME_UserDefinedBaseVoltage": cyme_base_voltage,
                "CYME_OperatingVoltage_LN": cyme_operating_kv,
                "VM_PU": vm_pu if is_source else "",
            }
        )

    df_buses = pd.DataFrame(bus_rows)

    def _split_hv_lv_rows(flat_df, ordered_df, hv_bus_idx, lv_bus_idx):
        """Return HV-side and LV-side row DataFrames.

        Preferred path uses bus ownership per row. Fallback uses MC ordering:
        first half rows are HV side, second half are LV side.
        """
        hv_side = pd.DataFrame(columns=flat_df.columns)
        lv_side = pd.DataFrame(columns=flat_df.columns)

        if "bus" in flat_df.columns:
            bus_num = pd.to_numeric(flat_df["bus"], errors="coerce")
            hv_side = flat_df.loc[bus_num == int(hv_bus_idx)]
            lv_side = flat_df.loc[bus_num == int(lv_bus_idx)]

        if hv_side.empty or lv_side.empty:
            n = len(ordered_df)
            if n:
                split = max(1, n // 2)
                hv_side = ordered_df.iloc[:split].copy()
                lv_side = ordered_df.iloc[split:].copy() if n > split else ordered_df.iloc[-split:].copy()
                hv_side = hv_side.reset_index(drop=True)
                lv_side = lv_side.reset_index(drop=True)

        return hv_side, lv_side

    def _side_connection_from_phases(side_df):
        """Derive winding type from phase wiring only.

        Rule: to_phase == 0 -> Wye, to_phase != 0 -> Delta.
        """
        if side_df.empty or "to_phase" not in side_df.columns:
            return "?"

        to_vals = pd.to_numeric(side_df["to_phase"], errors="coerce").dropna().astype(int)
        if to_vals.empty:
            return "?"

        if (to_vals == 0).all():
            return "Y"
        if (to_vals != 0).all():
            return "D"
        return "?"

    def _mean_vn_kv(side_df):
        if side_df.empty or "vn_kv" not in side_df.columns:
            return 0.0
        vals = pd.to_numeric(side_df["vn_kv"], errors="coerce").dropna().astype(float)
        return float(vals.mean()) if not vals.empty else 0.0

    def _phases_to_letters(values):
        out = []
        for p in values:
            try:
                out.append(PHASE_INT_TO_LETTER.get(int(p), "?"))
            except Exception:
                out.append("?")
        return out

    # ------------------------------------------------------------------ #
    # 2. TRANSFORMER TABLE (uses cached flat DataFrames)                 #
    # ------------------------------------------------------------------ #
    trafo_rows = []
    tol_kv = 0.01

    for t_idx, (flat, buses, ordered_t_data) in trafo_flat_cache.items():
        hv_bus = buses[0]
        lv_bus = buses[1] if len(buses) > 1 else buses[0]

        first = flat.iloc[0]
        sn_mva = float(first.get("sn_mva", 0))
        vk_pct = float(first.get("vk_percent", 0))
        vkr_pct = float(first.get("vkr_percent", 0))
        tap_pos = float(first.get("tap_pos", 0))

        hv_vn_kv = (
            float(net.bus.loc[(hv_bus, 1)]["vn_kv"])
            if (hv_bus, 1) in net.bus.index
            else float(net.bus.loc[hv_bus].iloc[0]["vn_kv"])
        )
        lv_vn_kv = (
            float(net.bus.loc[(lv_bus, 1)]["vn_kv"])
            if (lv_bus, 1) in net.bus.index
            else float(net.bus.loc[lv_bus].iloc[0]["vn_kv"])
        )

        hv_side, lv_side = _split_hv_lv_rows(flat, ordered_t_data, hv_bus, lv_bus)

        hv_conn_type = _side_connection_from_phases(hv_side)
        lv_conn_type = _side_connection_from_phases(lv_side)
        mc_connection = f"{hv_conn_type}_{lv_conn_type}"

        hv_winding = _mean_vn_kv(hv_side)
        lv_winding = _mean_vn_kv(lv_side)

        # Secondary voltage calculated from winding connection + winding voltage.
        if lv_conn_type == "Y":
            calculated_secondary_ll_kv = lv_winding * math.sqrt(3)
        elif lv_conn_type == "D":
            calculated_secondary_ll_kv = lv_winding
        else:
            calculated_secondary_ll_kv = np.nan

        cyme_secondary_ll_kv = lv_vn_kv

        calc_vs_bus_match = (
            abs(calculated_secondary_ll_kv - lv_vn_kv) <= tol_kv
            if pd.notna(calculated_secondary_ll_kv)
            else False
        )
        calc_vs_cyme_match = (
            abs(calculated_secondary_ll_kv - cyme_secondary_ll_kv) <= tol_kv
            if pd.notna(calculated_secondary_ll_kv)
            else False
        )
        secondary_voltage_all_match = calc_vs_bus_match and calc_vs_cyme_match

        # Keep old all-row views for compatibility with existing notebook outputs.
        from_phases = flat["from_phase"].values.tolist() if "from_phase" in flat.columns else []
        to_phases = flat["to_phase"].values.tolist() if "to_phase" in flat.columns else []

        tp_set = set(int(x) for x in pd.to_numeric(pd.Series(to_phases), errors="coerce").dropna().astype(int).tolist()) if to_phases else set()
        has_delta = any(tp != 0 for tp in tp_set)
        cyme_connection = "D_Yg" if has_delta else "Yg_Yg"

        cyme_z1_pct = vk_pct
        cyme_x1r1 = round(math.sqrt(max(0, (vk_pct / vkr_pct) ** 2 - 1)), 4) if vkr_pct > 0 else 10.0
        cyme_rating_kva = sn_mva * 1000
        mc_total_sn_mva = sn_mva * len(flat) / len(buses)

        trafo_rows.append(
            {
                "CKT_KEY": circuit_name,
                "TRAFO_IDX": t_idx,
                "HV_BUS": hv_bus,
                "LV_BUS": lv_bus,
                "HV_BUS_VN_KV": hv_vn_kv,
                "LV_BUS_VN_KV": lv_vn_kv,
                "MC_HV_WINDING_KV": hv_winding,
                "MC_LV_WINDING_KV": lv_winding,
                "MC_CONNECTION (derived)": mc_connection,
                "CYME_CONNECTION": cyme_connection,
                "CONNECTION_MATCH": mc_connection.replace("_", "") == cyme_connection.replace("_", "").replace("g", ""),
                "MC_HV_FROM_PHASES": _phases_to_letters(hv_side["from_phase"].tolist()) if "from_phase" in hv_side.columns else [],
                "MC_HV_TO_PHASES": _phases_to_letters(hv_side["to_phase"].tolist()) if "to_phase" in hv_side.columns else [],
                "MC_LV_FROM_PHASES": _phases_to_letters(lv_side["from_phase"].tolist()) if "from_phase" in lv_side.columns else [],
                "MC_LV_TO_PHASES": _phases_to_letters(lv_side["to_phase"].tolist()) if "to_phase" in lv_side.columns else [],
                "MC_FROM_PHASES": _phases_to_letters(from_phases),
                "MC_TO_PHASES": _phases_to_letters(to_phases),
                "MC_SN_MVA_PER_CIRCUIT": sn_mva,
                "MC_TOTAL_SN_MVA": mc_total_sn_mva,
                "MC_VK_PCT (per side)": vk_pct,
                "MC_VKR_PCT (per side)": vkr_pct,
                "CYME_Z1_PCT (fed as-is from mc)": cyme_z1_pct,
                "CYME_X1R1_RATIO": cyme_x1r1,
                "CYME_PrimaryV (L-L)": hv_vn_kv,
                "CYME_SecondaryV (L-L)": cyme_secondary_ll_kv,
                "CALCULATED_SecondaryV (L-L)": calculated_secondary_ll_kv,
                "CALC_vs_BUS_MATCH": calc_vs_bus_match,
                "CALC_vs_CYME_MATCH": calc_vs_cyme_match,
                "SECONDARY_VOLTAGE_ALL_MATCH": secondary_voltage_all_match,
                "CYME_RatingKVA": cyme_rating_kva,
                "MC_TAP_POS": tap_pos,
                "N_CIRCUITS": len(flat),
            }
        )

    df_trafos = pd.DataFrame(trafo_rows)
    return df_buses, df_trafos


# --- Run for all validation circuits ---
pkl_dir = Path(r"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final")
pkl_files = sorted(pkl_dir.glob("*.pkl"))
total_files = len(pkl_files)

bus_comparison_all = []
trafo_comparison_all = []

t_start = time.time()
for i, pkl_file in enumerate(pkl_files, 1):
    circuit_key = pkl_file.stem
    t0 = time.time()
    with open(pkl_file, "rb") as fh:
        net = pickle.load(fh)

    df_b, df_t = compare_bus_voltages_and_transformers(net, circuit_key)
    bus_comparison_all.append(df_b)
    trafo_comparison_all.append(df_t)

    elapsed = time.time() - t0
    total_elapsed = time.time() - t_start
    avg_per = total_elapsed / i
    eta = avg_per * (total_files - i)
    print(
        f"[{i}/{total_files}] {circuit_key}: {len(df_b)} buses, {len(df_t)} trafos  "
        f"({elapsed:.1f}s, ETA {eta:.0f}s)"
    )

df_all_buses = pd.concat(bus_comparison_all, ignore_index=True) if bus_comparison_all else pd.DataFrame()
df_all_trafos = pd.concat(trafo_comparison_all, ignore_index=True) if trafo_comparison_all else pd.DataFrame()

print(
    f"\nDone in {time.time() - t_start:.1f}s - "
    f"{len(df_all_buses)} bus entries, {len(df_all_trafos)} transformer entries across {len(bus_comparison_all)} circuits"
)

[1/882] CKT_100_16795: 2067 buses, 374 trafos  (1.8s, ETA 1565s)
[2/882] CKT_1014_17972: 3361 buses, 681 trafos  (2.6s, ETA 1915s)
[3/882] CKT_1019_17995: 1578 buses, 293 trafos  (1.1s, ETA 1609s)
[4/882] CKT_1021_18000: 1518 buses, 274 trafos  (1.0s, ETA 1419s)
[5/882] CKT_1025_18018: 1866 buses, 308 trafos  (1.1s, ETA 1328s)
[6/882] CKT_1029_18017: 1002 buses, 167 trafos  (0.6s, ETA 1198s)
[7/882] CKT_1031_18043: 523 buses, 17 trafos  (0.2s, ETA 1052s)
[8/882] CKT_1033_18045: 1705 buses, 304 trafos  (1.0s, ETA 1024s)
[9/882] CKT_1040_18076: 540 buses, 89 trafos  (0.4s, ETA 947s)
[10/882] CKT_1043_18094: 199 buses, 8 trafos  (0.1s, ETA 858s)
[11/882] CKT_1048_13343: 1503 buses, 279 trafos  (0.9s, ETA 851s)
[12/882] CKT_1051_12960: 2562 buses, 425 trafos  (1.6s, ETA 896s)
[13/882] CKT_1053_12976: 2203 buses, 332 trafos  (1.4s, ETA 923s)
[14/882] CKT_1074_13087: 193 buses, 10 trafos  (0.1s, ETA 862s)
[15/882] CKT_1093_15603: 1731 buses, 323 trafos  (1.4s, ETA 883s)
[16/882] CKT_1096_156

In [3]:
# --- Show buses where CYME would NOT have an explicit base voltage ---
# These are buses that are neither source nor transformer LV buses.
# CYME inherits voltage from upstream; mc always uses explicit vn_kv.
unset_buses = df_all_buses[df_all_buses['CYME_UserDefinedBaseVoltage'] == 'inherited/unset']
print(f"Buses WITHOUT explicit CYME UserDefinedBaseVoltage: {len(unset_buses)} / {len(df_all_buses)}")
print(f"Buses WITH explicit CYME voltage (source + trafo LV): {len(df_all_buses) - len(unset_buses)}")
print()

# Show per-circuit summary
voltage_summary = df_all_buses.groupby('CKT_KEY').agg(
    n_buses=('BUS_IDX', 'count'),
    n_source=('IS_SOURCE', 'sum'),
    n_trafo_lv=('IS_TRAFO_LV', 'sum'),
    n_cyme_unset=('CYME_UserDefinedBaseVoltage', lambda x: (x == 'inherited/unset').sum()),
    voltage_levels=('MC_VN_KV (L-L)', lambda x: sorted(x.unique().tolist())),
).reset_index()
voltage_summary

Buses WITHOUT explicit CYME UserDefinedBaseVoltage: 1151750 / 1373822
Buses WITH explicit CYME voltage (source + trafo LV): 222072



,CKT_KEY,n_buses,n_source,n_trafo_lv,n_cyme_unset,voltage_levels
0,CKT_100_16795,2067,1,374,1692,"[0.12, 0.24, 0.277, 12.0]"
1,CKT_1014_17972,3361,1,681,2679,"[0.12, 0.24, 0.277, 12.0]"
2,CKT_1019_17995,1578,1,293,1284,"[0.12, 0.24, 0.277, 12.0]"
3,CKT_1021_18000,1518,1,274,1243,"[0.12, 0.24, 0.277, 2.4, 12.0]"
4,CKT_1025_18018,1866,1,308,1557,"[0.12, 0.24, 0.277, 12.0]"
...,...,...,...,...,...,...
877,CKT_9177_03165,1107,1,130,976,"[0.12, 0.277, 12.0]"
878,CKT_9208_09357,1611,1,240,1370,"[0.104, 0.12, 0.277, 12.0]"
879,CKT_922_17915,4270,1,777,3492,"[0.12, 0.24, 0.277, 12.0]"
880,CKT_925_17912,814,1,124,689,"[0.12, 0.24, 0.277, 12.0]"


In [4]:
# --- Transformer configuration comparison ---
print("=== TRANSFORMER CONNECTION TYPE COMPARISON ===\n")

if not df_all_trafos.empty:
    # Ensure calculated-secondary comparison fields are present even if the main build cell was interrupted.
    if 'CALCULATED_TRANSFORMER_SECONDARY_LL_KV' not in df_all_trafos.columns:
        if 'CALCULATED_SecondaryV (L-L)' in df_all_trafos.columns:
            df_all_trafos['CALCULATED_TRANSFORMER_SECONDARY_LL_KV'] = df_all_trafos['CALCULATED_SecondaryV (L-L)']
        else:
            lv_conn = df_all_trafos['MC_CONNECTION (derived)'].astype(str).str.split('_').str[-1]
            lv_w = pd.to_numeric(df_all_trafos.get('MC_LV_WINDING_KV', np.nan), errors='coerce')
            df_all_trafos['CALCULATED_TRANSFORMER_SECONDARY_LL_KV'] = np.where(
                lv_conn.eq('Y'), lv_w * np.sqrt(3), np.where(lv_conn.eq('D'), lv_w, np.nan)
            )

    if 'CYME_SecondaryV (L-L)' not in df_all_trafos.columns:
        df_all_trafos['CYME_SecondaryV (L-L)'] = pd.to_numeric(df_all_trafos.get('LV_BUS_VN_KV', np.nan), errors='coerce')

    calc_sec = pd.to_numeric(df_all_trafos['CALCULATED_TRANSFORMER_SECONDARY_LL_KV'], errors='coerce')
    bus_sec = pd.to_numeric(df_all_trafos.get('LV_BUS_VN_KV', np.nan), errors='coerce')
    cyme_sec = pd.to_numeric(df_all_trafos.get('CYME_SecondaryV (L-L)', np.nan), errors='coerce')

    df_all_trafos['DELTA_CALC_TO_BUS_KV'] = calc_sec - bus_sec
    df_all_trafos['DELTA_CALC_TO_CYME_KV'] = calc_sec - cyme_sec

    tol_kv = 0.01
    df_all_trafos['CALC_vs_BUS_MATCH'] = calc_sec.notna() & bus_sec.notna() & (df_all_trafos['DELTA_CALC_TO_BUS_KV'].abs() <= tol_kv)
    df_all_trafos['CALC_vs_CYME_MATCH'] = calc_sec.notna() & cyme_sec.notna() & (df_all_trafos['DELTA_CALC_TO_CYME_KV'].abs() <= tol_kv)
    df_all_trafos['SECONDARY_VOLTAGE_ALL_MATCH'] = df_all_trafos['CALC_vs_BUS_MATCH'] & df_all_trafos['CALC_vs_CYME_MATCH']

    # Show connection mismatches (mc derived vs CYME simplified)
    mismatches = df_all_trafos[df_all_trafos['CONNECTION_MATCH'] == False]
    print(f"Connection type mismatches: {len(mismatches)} / {len(df_all_trafos)} transformers")
    if not mismatches.empty:
        print("\nMismatched transformers:")
        base_cols = ['CKT_KEY', 'TRAFO_IDX', 'MC_CONNECTION (derived)', 'CYME_CONNECTION']
        phase_cols = [c for c in ['MC_HV_TO_PHASES', 'MC_LV_TO_PHASES', 'MC_FROM_PHASES', 'MC_TO_PHASES'] if c in mismatches.columns]
        print(mismatches[base_cols + phase_cols].to_string(index=False))

    print(f"\n\n=== SECONDARY VOLTAGE MATCH CHECK ===")
    v_mismatch = df_all_trafos[df_all_trafos['SECONDARY_VOLTAGE_ALL_MATCH'] == False]
    print(f"Secondary voltage mismatches: {len(v_mismatch)} / {len(df_all_trafos)} transformers")
    if not v_mismatch.empty:
        print(v_mismatch[['CKT_KEY', 'TRAFO_IDX', 'LV_BUS_VN_KV', 'MC_LV_WINDING_KV',
                          'CALCULATED_TRANSFORMER_SECONDARY_LL_KV', 'CYME_SecondaryV (L-L)',
                          'DELTA_CALC_TO_BUS_KV', 'DELTA_CALC_TO_CYME_KV',
                          'CALC_vs_BUS_MATCH', 'CALC_vs_CYME_MATCH']].head(50).to_string(index=False))

    print(f"\n\n=== CYME CONNECTION DISTRIBUTION ===")
    print(df_all_trafos['CYME_CONNECTION'].value_counts())
    print(f"\n=== MC CONNECTION DISTRIBUTION ===")
    print(df_all_trafos['MC_CONNECTION (derived)'].value_counts())

    # Impedance comparison: mc stores half-impedance per side,
    # but mc_to_cyme.py feeds vk_percent directly as CYME Z1%.
    # CYME expects TOTAL positive-sequence impedance.
    # If mc has vk/2 and CYME gets vk/2 as "total", the impedance is HALVED.
    print(f"\n\n=== IMPEDANCE ANALYSIS ===")
    print("MC stores vk_percent as HALF the total (split between HV and LV sides).")
    print("mc_to_cyme.py feeds this half-value directly to CYME as PositiveSequenceImpedancePercent.")
    print("This means CYME sees HALF the actual transformer impedance!\n")

    imp_summary = df_all_trafos[['CKT_KEY', 'TRAFO_IDX', 'MC_VK_PCT (per side)',
                                 'CYME_Z1_PCT (fed as-is from mc)', 'MC_TOTAL_SN_MVA', 'CYME_RatingKVA']].copy()
    imp_summary['CORRECT_TOTAL_Z1_PCT'] = imp_summary['MC_VK_PCT (per side)'] * 2
    imp_summary['Z_ERROR_FACTOR'] = 0.5  # CYME gets half the impedance
    imp_summary
else:
    print("No transformers found in the loaded networks.")

=== TRANSFORMER CONNECTION TYPE COMPARISON ===

Connection type mismatches: 23950 / 221201 transformers

Mismatched transformers:
           CKT_KEY  TRAFO_IDX MC_CONNECTION (derived) CYME_CONNECTION MC_HV_TO_PHASES MC_LV_TO_PHASES     MC_FROM_PHASES       MC_TO_PHASES
     CKT_100_16795         49                     Y_D            D_Yg       [N, N, N]       [B, C, A] [A, B, C, A, B, C] [B, C, A, N, N, N]
     CKT_100_16795         59                     D_D            D_Yg       [B, C, A]       [B, C, A] [A, B, C, A, B, C] [B, C, A, B, C, A]
     CKT_100_16795         60                     Y_D            D_Yg       [N, N, N]       [B, C, A] [A, B, C, A, B, C] [B, C, A, N, N, N]
     CKT_100_16795         63                     D_D            D_Yg       [B, C, A]       [B, C, A] [A, B, C, A, B, C] [B, C, A, B, C, A]
     CKT_100_16795         65                     Y_D            D_Yg       [N, N, N]       [B, C, A] [A, B, C, A, B, C] [B, C, A, N, N, N]
     CKT_100_16795         69 

In [5]:
v_mismatch

,CKT_KEY,TRAFO_IDX,HV_BUS,LV_BUS,HV_BUS_VN_KV,LV_BUS_VN_KV,MC_HV_WINDING_KV,MC_LV_WINDING_KV,MC_CONNECTION (derived),CYME_CONNECTION,...,CALCULATED_SecondaryV (L-L),CALC_vs_BUS_MATCH,CALC_vs_CYME_MATCH,SECONDARY_VOLTAGE_ALL_MATCH,CYME_RatingKVA,MC_TAP_POS,N_CIRCUITS,CALCULATED_TRANSFORMER_SECONDARY_LL_KV,DELTA_CALC_TO_BUS_KV,DELTA_CALC_TO_CYME_KV
63,CKT_100_16795,63,841,1394,12.0,0.12,12.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,16.666667,0.0,6,0.069282,-0.050718,-0.050718
71,CKT_100_16795,71,929,1923,12.0,0.12,12.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,6.666667,0.0,6,0.069282,-0.050718,-0.050718
72,CKT_100_16795,72,1127,1139,12.0,0.12,12.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,16.666667,0.0,6,0.069282,-0.050718,-0.050718
73,CKT_100_16795,73,155,654,12.0,0.12,12.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,16.666667,0.0,6,0.069282,-0.050718,-0.050718
86,CKT_100_16795,86,743,1872,12.0,0.12,12.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,16.666667,0.0,6,0.069282,-0.050718,-0.050718
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221169,CKT_92_16775,127,420,522,16.0,0.12,16.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,10.000000,0.0,6,0.069282,-0.050718,-0.050718
221170,CKT_92_16775,128,584,610,16.0,0.12,16.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,25.000000,0.0,6,0.069282,-0.050718,-0.050718
221182,CKT_92_16775,140,286,372,16.0,0.12,16.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,25.000000,0.0,6,0.069282,-0.050718,-0.050718
221183,CKT_92_16775,141,731,932,16.0,0.12,16.0,0.069282,D_D,D_Yg,...,0.069282,False,False,False,15.000000,0.0,6,0.069282,-0.050718,-0.050718


In [16]:
df_all_trafos

,CKT_KEY,TRAFO_IDX,HV_BUS,LV_BUS,HV_BUS_VN_KV,LV_BUS_VN_KV,MC_HV_WINDING_KV,MC_LV_WINDING_KV,MC_CONNECTION (derived),CYME_CONNECTION,...,CALCULATED_SecondaryV (L-L),CALC_vs_BUS_MATCH,CALC_vs_CYME_MATCH,SECONDARY_VOLTAGE_ALL_MATCH,CYME_RatingKVA,MC_TAP_POS,N_CIRCUITS,CALCULATED_TRANSFORMER_SECONDARY_LL_KV,DELTA_CALC_TO_BUS_KV,DELTA_CALC_TO_CYME_KV
0,CKT_100_16795,0,385,804,0.120,12.00,0.069282,6.928200,Y_Y,Yg_Yg,...,11.999994,True,True,True,37.500000,0.0,4,11.999994,-0.000006,-0.000006
1,CKT_100_16795,1,223,1395,0.120,12.00,0.069282,6.928200,Y_Y,Yg_Yg,...,11.999994,True,True,True,37.500000,0.0,4,11.999994,-0.000006,-0.000006
2,CKT_100_16795,2,1245,1411,12.000,0.12,6.928200,0.069282,Y_Y,Yg_Yg,...,0.120000,True,True,True,37.500000,0.0,4,0.120000,0.000000,0.000000
3,CKT_100_16795,3,1168,1800,12.000,0.12,6.928200,0.069282,Y_Y,Yg_Yg,...,0.120000,True,True,True,12.500000,0.0,4,0.120000,0.000000,0.000000
4,CKT_100_16795,4,1848,2009,0.277,12.00,0.159926,6.928200,Y_Y,Yg_Yg,...,11.999994,True,True,True,12.500000,0.0,4,11.999994,-0.000006,-0.000006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221196,CKT_92_16775,154,533,674,16.000,0.12,9.237600,0.069282,Y_Y,Yg_Yg,...,0.120000,True,True,True,7.500000,0.0,4,0.120000,0.000000,0.000000
221197,CKT_92_16775,155,315,367,0.120,16.00,0.069282,9.237600,Y_Y,Yg_Yg,...,15.999993,True,True,True,25.000000,0.0,4,15.999993,-0.000007,-0.000007
221198,CKT_92_16775,156,93,484,0.120,16.00,0.069282,9.237600,Y_Y,Yg_Yg,...,15.999993,True,True,True,25.000000,0.0,4,15.999993,-0.000007,-0.000007
221199,CKT_92_16775,157,688,951,16.000,0.12,9.237600,0.069282,Y_Y,Yg_Yg,...,0.120000,True,True,True,7.500000,0.0,4,0.120000,0.000000,0.000000


In [ ]:
# --- Voltage discrepancy analysis: buses with different voltage levels ---
# Vectorized summary instead of per-circuit print loop

print("=== BUS VOLTAGE LEVEL ANALYSIS ===\n")

# Per-circuit summary via groupby (fast)
voltage_analysis = df_all_buses.groupby('CKT_KEY').agg(
    n_buses=('BUS_IDX', 'count'),
    n_source=('IS_SOURCE', 'sum'),
    n_trafo_lv=('IS_TRAFO_LV', 'sum'),
    voltage_levels=('MC_VN_KV (L-L)', lambda x: sorted(x.unique().tolist())),
    n_voltage_levels=('MC_VN_KV (L-L)', 'nunique'),
    source_voltage=('MC_VN_KV (L-L)', lambda x: x[df_all_buses.loc[x.index, 'IS_SOURCE']].unique().tolist()),
).reset_index()

# Regular (non-source, non-trafo-LV) bus voltage distribution per circuit
regular_mask = ~df_all_buses['IS_SOURCE'] & ~df_all_buses['IS_TRAFO_LV']
regular_vcount = (
    df_all_buses[regular_mask]
    .groupby(['CKT_KEY', 'MC_VN_KV (L-L)'])
    .size()
    .reset_index(name='count')
    .groupby('CKT_KEY')
    .apply(lambda g: dict(zip(g['MC_VN_KV (L-L)'], g['count'])), include_groups=False)
    .rename('regular_buses_by_voltage')
)
voltage_analysis = voltage_analysis.merge(regular_vcount, on='CKT_KEY', how='left')

# Trafo LV bus details per circuit (replace iterrows with a grouped join)
trafo_lv_details = (
    df_all_buses[df_all_buses['IS_TRAFO_LV']]
    .groupby('CKT_KEY')
    .apply(
        lambda g: [
            f"{r['BUS_NAME']}(idx={r['BUS_IDX']}, {r['MC_VN_KV (L-L)']}kV, {r['TRAFO_NAME']})"
            for _, r in g[['BUS_NAME', 'BUS_IDX', 'MC_VN_KV (L-L)', 'TRAFO_NAME']].iterrows()
        ],
        include_groups=False,
    )
    .rename('trafo_lv_buses')
)
voltage_analysis = voltage_analysis.merge(trafo_lv_details, on='CKT_KEY', how='left')

print(f"Total circuits: {len(voltage_analysis)}")
print(f"Circuits with >2 voltage levels: {(voltage_analysis['n_voltage_levels'] > 2).sum()}")
print(f"Circuits with trafo LV buses: {(voltage_analysis['n_trafo_lv'] > 0).sum()}")
voltage_analysis

In [11]:
# --- Full transformer table for inspection ---
if not df_all_trafos.empty:
    display_cols = [
        'CKT_KEY', 'TRAFO_IDX', 'HV_BUS', 'LV_BUS',
        'HV_BUS_VN_KV', 'LV_BUS_VN_KV',
        'MC_HV_WINDING_KV', 'MC_LV_WINDING_KV',
        'MC_CONNECTION (derived)', 'CYME_CONNECTION', 'CONNECTION_MATCH',
        'MC_HV_FROM_PHASES', 'MC_HV_TO_PHASES', 'MC_LV_FROM_PHASES', 'MC_LV_TO_PHASES',
        'CALCULATED_TRANSFORMER_SECONDARY_LL_KV', 'CYME_SecondaryV (L-L)',
        'DELTA_CALC_TO_BUS_KV', 'DELTA_CALC_TO_CYME_KV',
        'CALC_vs_BUS_MATCH', 'CALC_vs_CYME_MATCH', 'SECONDARY_VOLTAGE_ALL_MATCH',
        'MC_SN_MVA_PER_CIRCUIT', 'MC_TOTAL_SN_MVA',
        'MC_VK_PCT (per side)', 'MC_VKR_PCT (per side)',
        'CYME_Z1_PCT (fed as-is from mc)', 'CYME_X1R1_RATIO',
        'CYME_RatingKVA', 'MC_TAP_POS',
    ]
    display_cols = [c for c in display_cols if c in df_all_trafos.columns]
    df_all_trafos[display_cols]

In [ ]:
# --- Power flow settings comparison: key differences that affect results ---
print("=" * 80)
print("POWER FLOW SETTINGS COMPARISON: MC vs CYME")
print("=" * 80)

differences = [
    ("Line Charging",
     "INCLUDED in Y_network admittance matrix",
     "DISABLED (IncludeLineCharging=0)",
     "CYME ignores shunt capacitance of lines → lower reactive power injection → different voltage profile"),

    ("Voltage-Sensitive Load Model",
     "Supports const_z_percent and const_i_percent per load (ZIP model)",
     "DISABLED (VoltageSensitivityLoadModel.V=0 → constant P/Q only)",
     "MC loads may have voltage-dependent behavior; CYME treats all as constant PQ"),

    ("Source Impedance",
     "Included via ext_grid_sequence R/X values in Y_source",
     "DISABLED (IncludeSourceImpedance=0)",
     "CYME treats source as ideal → source bus voltage is exact setpoint"),

    ("Line Transposition",
     "Full phase-unbalanced matrices (no transposition)",
     "DISABLED (AssumeLineTransposition=0)",
     "Both use untranspositioned lines — this is consistent"),

    ("Transformer Impedance",
     "vk_percent stored as HALF total (split HV/LV sides)",
     "Receives vk_percent directly as PositiveSequenceImpedancePercent",
     "⚠️ POSSIBLE BUG: CYME may be getting half the correct impedance if it expects total Z%"),

    ("Transformer Connection",
     "Full phase mapping via from_phase/to_phase arrays per circuit",
     "Simplified to D_Yg or Yg_Yg based on presence of non-zero to_phase",
     "Connection logic may not capture all vector group variations correctly"),

    ("Capacitor/Regulator Controls",
     "Can run iterative control loop (run_control=True)",
     "ALL controls LOCKED by default (LockSwitchedCapacitors=1, etc.)",
     "CYME locks cap banks and regulators at initial state"),

    ("Bus Base Voltage Assignment",
     "Explicit vn_kv per bus in net.bus table (set at creation time)",
     "UserDefinedBaseVoltage set only on source and transformer LV nodes; rest inherited",
     "CYME engine propagates voltage through topology; MC relies on explicit assignment"),
]

for name, mc_val, cyme_val, impact in differences:
    print(f"\n{'─' * 80}")
    print(f"  {name}")
    print(f"  MC:   {mc_val}")
    print(f"  CYME: {cyme_val}")
    print(f"  Impact: {impact}")

In [ ]:
# --- Display full bus table with voltage comparison ---
df_all_buses

In [4]:
# Quick quality check for derived MC connections
q_count = int(df_all_trafos['MC_CONNECTION (derived)'].str.contains(r'\?').sum()) if not df_all_trafos.empty else 0
print(f"Rows with '?' in MC_CONNECTION (derived): {q_count} / {len(df_all_trafos)}")
print(df_all_trafos['MC_CONNECTION (derived)'].value_counts().head(10))

Rows with '?' in MC_CONNECTION (derived): 0 / 221201
MC_CONNECTION (derived)
Y_Y    188002
Y_D     16602
D_Y     16581
D_D        16
Name: count, dtype: int64


## Update MC Transformer Configurations to Match CYME

CYME only supports **Y-Y** and **D-Y** connections. This cell converts all MC networks:
- **Y_D → Y_Y**: Change LV Delta to Wye
- **D_D → D_Y**: Change LV Delta to Wye

When converting LV from **Delta → Wye**:
1. Set `to_phase = 0` (neutral) on all LV-side rows
2. Set `from_phase = [1, 2, 3]` (standard Wye group-0 mapping)
3. Divide LV-side `vn_kv` by √3 (Delta stores L-L voltage directly; Wye stores L-L/√3)

Updated networks are saved as `.pkl` files in `networks/mc_xfmr/`.

In [1]:
import pickle, math, time, copy
from pathlib import Path
import pandas as pd
import numpy as np


def update_mc_xfmr_connections(net):
    """Update transformer configurations in an MC net so that all secondaries
    are Wye (matching CYME's Y-Y / D-Y convention).

    Transformations applied:
        Y_D  ->  Y_Y   (LV Delta -> Wye, divide LV vn_kv by sqrt(3))
        D_D  ->  D_Y   (LV Delta -> Wye, divide LV vn_kv by sqrt(3))
        Y_Y  ->  unchanged
        D_Y  ->  unchanged

    Returns a dict summarising changes: {trafo_idx: {'old': str, 'new': str}}.
    The net object is modified **in-place**.
    """
    SQRT3 = math.sqrt(3)
    changes = {}

    if "trafo1ph" not in net or net["trafo1ph"].empty:
        return changes

    trafo = net.trafo1ph
    trafo_indices = trafo.index.get_level_values(0).unique()

    for t_idx in trafo_indices:
        t_data = trafo.loc[t_idx]
        if isinstance(t_data, pd.Series):
            t_data = t_data.to_frame().T

        flat = t_data.reset_index() if isinstance(t_data.index, pd.MultiIndex) else t_data

        # Identify buses (HV = lower index, LV = higher index)
        if "bus" in flat.columns:
            bus_values = pd.to_numeric(flat["bus"], errors="coerce").dropna().astype(int).tolist()
        else:
            bus_values = flat.index.get_level_values("bus").astype(int).tolist()
        buses = sorted(set(bus_values))
        if len(buses) < 2:
            continue
        hv_bus, lv_bus = buses[0], buses[1]

        # Determine LV-side connection from to_phase
        if "bus" in flat.columns:
            lv_mask_flat = pd.to_numeric(flat["bus"], errors="coerce").astype(int) == lv_bus
        else:
            lv_mask_flat = flat.index.get_level_values("bus").astype(int) == lv_bus
        lv_to_phases = pd.to_numeric(flat.loc[lv_mask_flat, "to_phase"], errors="coerce").dropna().astype(int)
        if lv_to_phases.empty:
            continue

        lv_is_delta = (lv_to_phases != 0).all()
        if not lv_is_delta:
            continue  # Already Wye — nothing to do

        # Determine HV-side connection for logging
        if "bus" in flat.columns:
            hv_mask_flat = pd.to_numeric(flat["bus"], errors="coerce").astype(int) == hv_bus
        else:
            hv_mask_flat = flat.index.get_level_values("bus").astype(int) == hv_bus
        hv_to_phases = pd.to_numeric(flat.loc[hv_mask_flat, "to_phase"], errors="coerce").dropna().astype(int)
        hv_conn = "D" if (hv_to_phases != 0).all() else "Y"
        old_conn = f"{hv_conn}_D"
        new_conn = f"{hv_conn}_Y"

        # --- Apply changes to the real (multi-indexed) trafo1ph DataFrame ---
        # Build a boolean mask over the full trafo1ph index for this transformer's LV rows
        idx = trafo.index
        level0 = idx.get_level_values(0)  # trafo index
        level1 = idx.get_level_values(1)  # bus
        lv_mask = (level0 == t_idx) & (level1.astype(int) == lv_bus)

        # 1. Set to_phase = 0 (Wye: connect to neutral)
        trafo.loc[lv_mask, "to_phase"] = 0

        # 2. Set from_phase = cycling 1,2,3 per circuit (standard Y group-0)
        n_lv_rows = int(lv_mask.sum())
        y_from_phases = [1, 2, 3] * max(1, n_lv_rows // 3)
        trafo.loc[lv_mask, "from_phase"] = y_from_phases[:n_lv_rows]

        # 3. Divide vn_kv by sqrt(3): Delta stored L-L, Wye stores L-L/sqrt(3)
        trafo.loc[lv_mask, "vn_kv"] = trafo.loc[lv_mask, "vn_kv"].astype(float) / SQRT3

        changes[t_idx] = {"old": old_conn, "new": new_conn}

    return changes


# ── Run across all MC networks and save updated copies ───────────────────────
src_dir = Path(r"networks/mc")
dst_dir = Path(r"networks/mc_xfmr")
dst_dir.mkdir(parents=True, exist_ok=True)

pkl_files = sorted(src_dir.glob("*.pkl"))
total = len(pkl_files)
summary_rows = []

t_start = time.time()
for i, pkl_path in enumerate(pkl_files, 1):
    circuit_key = pkl_path.stem
    t0 = time.time()

    with open(pkl_path, "rb") as fh:
        net = pickle.load(fh)

    changes = update_mc_xfmr_connections(net)

    # Save updated network
    out_path = dst_dir / pkl_path.name
    with open(out_path, "wb") as fh:
        pickle.dump(net, fh)

    for t_idx, info in changes.items():
        summary_rows.append({"CKT_KEY": circuit_key, "TRAFO_IDX": t_idx,
                             "OLD_CONN": info["old"], "NEW_CONN": info["new"]})

    elapsed = time.time() - t0
    total_elapsed = time.time() - t_start
    avg = total_elapsed / i
    eta = avg * (total - i)
    if changes:
        print(f"[{i}/{total}] {circuit_key}: updated {len(changes)} trafo(s) "
              f"({elapsed:.1f}s, ETA {eta:.0f}s)")

total_time = time.time() - t_start
df_changes = pd.DataFrame(summary_rows)
print(f"\nDone in {total_time:.1f}s — {len(df_changes)} transformers updated "
      f"across {total} circuits.")
if not df_changes.empty:
    print("\nChange summary:")
    print(df_changes.groupby(["OLD_CONN", "NEW_CONN"]).size().reset_index(name="COUNT").to_string(index=False))

[1/882] CKT_100_16795: updated 94 trafo(s) (4.0s, ETA 3493s)
[2/882] CKT_1014_17972: updated 41 trafo(s) (0.8s, ETA 2098s)
[3/882] CKT_1019_17995: updated 4 trafo(s) (0.4s, ETA 1500s)
[4/882] CKT_1021_18000: updated 21 trafo(s) (0.4s, ETA 1214s)
[5/882] CKT_1025_18018: updated 22 trafo(s) (0.4s, ETA 1042s)
[6/882] CKT_1029_18017: updated 21 trafo(s) (0.2s, ETA 899s)
[7/882] CKT_1031_18043: updated 6 trafo(s) (0.0s, ETA 774s)
[8/882] CKT_1033_18045: updated 23 trafo(s) (0.3s, ETA 714s)
[9/882] CKT_1040_18076: updated 37 trafo(s) (0.1s, ETA 648s)
[10/882] CKT_1043_18094: updated 3 trafo(s) (0.0s, ETA 584s)
[11/882] CKT_1048_13343: updated 16 trafo(s) (0.3s, ETA 555s)
[12/882] CKT_1051_12960: updated 28 trafo(s) (0.5s, ETA 548s)
[13/882] CKT_1053_12976: updated 7 trafo(s) (0.4s, ETA 529s)
[14/882] CKT_1074_13087: updated 1 trafo(s) (0.0s, ETA 492s)
[15/882] CKT_1093_15603: updated 33 trafo(s) (0.4s, ETA 480s)
[16/882] CKT_1096_15608: updated 1 trafo(s) (0.2s, ETA 458s)
[17/882] CKT_1105_1